### 살시간 전력 현황
문제: API 동기화 까지 2시간 - 최대 1일 걸림 -> 그래서 현재 데이터 불러올 수 없음

In [109]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

API = "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b"

url = "https://openapi.kpx.or.kr/openapi/sukub5mToday/getSukub5mToday"

params = {
    "serviceKey": API
}

response = requests.get(url, params=params)
content = response.content.decode("utf-8")
content

'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>30</resultCode><resultMsg>SERVICE KEY IS NOT REGISTERED ERROR.</resultMsg><pageNo>1</pageNo><numOfRows>1000</numOfRows><totalCount>0</totalCount><pageSize>1</pageSize><startPage>1</startPage></header></response>'

In [ ]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET

API = "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b"

url = "https://openapi.kpx.or.kr/openapi/sukub5mToday/getSukub5mToday"

params = {
    "serviceKey": API
}

response = requests.get(url, params=params)
# response.raise_for_status()

content = response.content.decode("utf-8")


def kpx_response_to_dataframe(content: str) -> pd.DataFrame:
    root = ET.fromstring(content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")
    if result_code and result_code != "00":
        raise ValueError(f"API error {result_code}: {result_msg or 'Unknown error'}")

    items = root.findall(".//item")
    rows = []

    for item in items:
        row = {
            child.tag: child.text
            for child in item
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    num_cols = [
        "suppAbility",
        "currPwrTot",
        "forecastLoad",
        "suppReservePwr",
        "suppReserveRate",
        "operReservePwr",
        "operReserveRate"
    ]

    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "baseDatetime" in df.columns:
        df["baseDatetime"] = pd.to_datetime(
            df["baseDatetime"],
            format="%Y%m%d%H%M%S",
            errors="coerce"
        )

    return df


df = kpx_response_to_dataframe(content)



ValueError: API error 30: SERVICE KEY IS NOT REGISTERED ERROR.

### 기상청 API
문제 : 서울만 가져올 수 없음

In [63]:
# 기상청 API 테스트 -> 서울만 아님(수도권)
api = "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b"


import requests

url = 'http://apis.data.go.kr/1360000/MidFcstInfoService/getMidFcst'
params ={'serviceKey' : "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b",
         'pageNo' : '1',
         'numOfRows' : '10',
         'dataType' : 'XML',
         'stnId' : '109',
         'tmFc' : '202606300600' }

response = requests.get(url, params=params)
content = response.content.decode("utf-8")

print(content)

<?xml version="1.0" encoding="UTF-8"?>
<response><header><resultCode>00</resultCode><resultMsg>NORMAL_SERVICE</resultMsg></header><body><dataType>XML</dataType><items><item><wfSv>○ (강수) 7월 6일(월)과 9일(목)은 비가 내리겠습니다.
○ (기온) 이번 예보기간 아침 기온은 21~24℃, 낮 기온은 27~32℃로 평년(최저기온 20~22℃, 최고기온 27~30℃)과 비슷하거나 조금 높겠습니다.
○ (해상) 서해중부해상의 물결은 0.5~2.0m로 일겠습니다.
○ (주말 전망) 7월 4일(토)과 5일(일)은 대체로 흐리겠습니다. 아침 기온은 21~23℃, 낮 기온은 29~32℃가 되겠습니다.

* (변동성) 이번 예보기간에는 북태평양고기압 가장자리의 위치 변화와 태풍 등 열대 요란의 발생이나 이동에 따른 우리나라 주변 기압계 변동성이 있겠고, 강수지역과 시점이 변경될 가능성이 있겠으니, 앞으로 발표하는 최신 예보를 참고하기 바랍니다.</wfSv></item></items><numOfRows>10</numOfRows><pageNo>1</pageNo><totalCount>1</totalCount></body></response>



### 기상청_지상(종관, ASOS) 시간자료 조회서비스
문제 : 실시간 아님 전날까지 자료만 있음

In [62]:
# 기상청_지상(종관, ASOS) 시간자료 조회서비스 -> 실시간 아님 전날 자료만 가능

import requests
import pandas as pd
import xml.etree.ElementTree as ET

API = "4898e1e9d1fb4330c32a1c4625a551fac391c606c58aab3614339ab86074969b"

url = "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"

params = {
    "serviceKey": API,
    "pageNo": "1",
    "numOfRows": "10",
    "dataType": "XML",
    "dataCd": "ASOS",
    "dateCd": "HR",
    "startDt": "20260630",
    "startHh": "00",
    "endDt": "20260630",
    "endHh": "13",
    "stnIds": "108" # 서울
}

response = requests.get(url, params=params)
response.raise_for_status()

xml = response.content.decode("utf-8")

root = ET.fromstring(xml)

# 오류 확인
result_code = root.findtext(".//resultCode")
result_msg = root.findtext(".//resultMsg")

print(result_code, result_msg)

items = root.findall(".//item")

rows = []

for item in items:
    row = {}

    for child in item:
        row[child.tag] = child.text

    rows.append(row)

df = pd.DataFrame(rows)

df.head()

99 전날 자료까지 제공됩니다. 날짜 범위를 확인해주세요.


""
